# Step.0 說明

> **本檔為 v9 的原始設計紀錄。** 消融實驗與後續長跑請改用同目錄的
> `train_ablation.ipynb`（改一行 `ARM` 就能切換六個臂），
> 判準與執行順序見 `docs/v9_實驗重整計畫.md`。
> Step.2 / Step.3 的實作已移到 `v9_modules.py`，本檔只負責取得與安裝。

### Version 9
- 新v5 Datasets
- YOLO26-Nano-P2（P2/P3/P4/P5 四尺度偵測頭）
- Wise-Inner-MPDIoU 自訂邊界框損失（取代 CIoU）
- StarTripletBlock（StarNet 星形升維 + Triplet Attention）強化淺層 P2 特徵
- ADown 雙路池化下採樣取代 stride-2 Conv

v9 以 v8 用的官方 `yolo26-p2.yaml` 為基準，**只動三處**，其餘層號、通道數、Detect 取用層完全對齊，
確保 mAP 差異可歸因到改動本身而非頭部配置：
| # | 位置 | v8 | v9 |
| --- | --- | --- | --- |
| 1 | 第 2 / 4 / 19 層 | `C3k2` | `StarTripletBlock` |
| 2 | 第 3 / 5 / 7 / 20 / 23 / 26 層 | stride-2 `Conv` | `ADown` |
| 3 | box loss | 內建 CIoU | Wise-Inner-MPDIoU |

### 與 v8 的對照基準
| 指標 | v8 (Epoch 69) |
| --- | --- |
| mAP@0.5 | 82.08% |
| mAP@0.5:0.95 | 64.69% |
| Scale_Insect FP / FPR | 615 / 39.30% |
| Thrips Recall | 53.91% |

### 本機實測成本（ultralytics 8.4.121, imgsz=640, nc=8, scale n）
| 模型 | 參數量 | GFLOPs | 活化元素 (B=1) |
| --- | ---: | ---: | ---: |
| v8（官方 yolo26-p2） | 2,517,824 | 7.65 | 102,660,800 |
| **v9（STB + ADown）** | **2,225,716** | **11.00** | **187,774,400** |
| 倍率 | 0.88x | 1.44x | 1.83x |

參數量反而少 12%（StarTripletBlock 主體是深度可分離卷積），但 FLOPs 與活化記憶體上升。
1-Epoch 實測：`batch=24` 時 VRAM 峰值 14.1G / 14.9G，沒有 OOM 但幾乎無餘裕，
160 epochs 的長跑已改用 **`batch=20`**（自動累積後的等效 batch 為 60，比 `batch=24` 的 72 更接近 v8 的 64）。

損失量級實測（同一批微小框，相對內建 CIoU）：
| λ_mpd / λ_inner | 損失均值 | 倍率 | `box=8.0` 的等效權重 |
| --- | ---: | ---: | ---: |
| 1.00 / 1.00（預設） | 1.2763 | 2.16x | 17.3 |
| 0.50 / 0.50 | 0.9131 | 1.55x | 12.4 |
| 0.50 / 0.25 | 0.7514 | 1.27x | 10.2 |
| CIoU（v8 基準） | 0.5895 | 1.00x | 8.0 |

### Kaggle 1-Epoch 實測結果（2026-08-19, T4 x1, v9_test01）

Step.1 ~ Step.4.5 全數通過，`model.info()` 印出 **2,225,716 params / 11.0 GFLOPs**，與本機完全一致。
1 個 epoch 正常完成，**沒有 NaN、沒有 OOM**，epoch 內 loss 單調下降
（cls 24.32 → 14.14、box 4.92 → 4.46）。

| 項目 | v8 (Epoch 1) | v9 (Epoch 1) | 說明 |
| --- | ---: | ---: | --- |
| box_loss | 3.11981 | 4.409 | 1.41x，與預測的 1.35x 相符（5.0x2.16 / 8.0x1.00） |
| cls_loss | 9.13032 | 13.76 | 1.51x，其中 1.25x 來自 `cls` 0.8→1.0 |
| l1_loss | 0.03421 | 0.03339 | 幾乎相同 |
| mAP50 | 0.21425 | 0.0758 | 見下方說明 |
| s/epoch | 157 | 355 | **2.26x** |
| VRAM 峰值 | — | 14.1G / 14.9G | 幾乎無餘裕 |

**mAP 落後的主因是預訓練權重轉移率**，不是模型有問題。官方 `yolo26n.pt` 的 backbone
第 0~10 層與 `yolo26-p2.yaml` 完全相同，所以 v8 的淺層 backbone 幾乎整段沿用；
v9 把第 2/3/4/5/7 層換成 StarTripletBlock 與 ADown 之後，這幾層只能從零開始學：

| 模型 | 可轉移張量 | 參數覆蓋率 |
| --- | ---: | ---: |
| v8 官方 yolo26-p2 | 360/902 (39.9%) | **60.3%** |
| v9 STB + ADown | 284/998 (28.5%) | **45.1%** |

（v9 的 284/998 與 Kaggle log 第 119 行完全吻合。）
換句話說 v9 是拿 45% 的預訓練基礎去跟 v8 的 60% 比第 1 輪，落後屬預期，
但也代表 **v9 需要更多 epoch 才追得上**，160 epochs 不宜再縮減。

**慢 2.26x 的主因**是 `TripletAttention` 在 P2（160x160）對 3 倍升維後的張量做
permute / contiguous，屬記憶體頻寬成本，GFLOPs 量不到（本機 fwd+bwd 實測 2.51x，
與 Kaggle 的 2.26x 相符）。已評估過把 TA 移到降維後可降到 1.89x，
但為維持 StarNet 原始設計不動，改以**分兩場 session** 完成 160 epochs（見 RESUME 段）。

### v9 修正紀錄（2026-08-19）
下列問題在首次執行前即已定位並修正，細節見各 Step 內註解：
1. **yaml 缺少 `end2end: True` 與 `reg_max: 1`** → 會靜默退回成非 E2E + DFL(reg_max=16) 架構，與 v8 不同體系、無法對照（v8 `results.csv` 有 `train/l1_loss` 欄可證 v8 為 E2E/reg_max=1）。
2. **自訂 `bbox_iou` 簽名不相容** → `ultralytics/utils/loss.py:133` 以 `bbox_iou(..., xywh=False, CIoU=True)` 呼叫，原函式無 `CIoU` 參數，第一個 batch 就 `TypeError`。
3. **自訂 `bbox_iou` 回傳形狀錯誤** → 原用 `box1[..., 0]` 索引會掉最後一維，回傳 `[N]`；`BboxLoss` 的 `weight` 為 `[N,1]`，相乘廣播成 `[N,N]`，損失完全錯誤且記憶體暴增。已改用 `chunk(4, -1)`。
4. **交集算式打錯** → `torch.max(b1_x1, b1_x1)` 兩個參數都是 `b1`。
5. **自訂 `ADown` 解析度不一致** → 先 `avg_pool` 再對其中一路 `max_pool`，兩路為 H/2 與 H/4，`torch.cat` 必定 `RuntimeError`。已改用 Ultralytics 內建 `ADown`。
6. **`StarTripletBlock` 無法被 `parse_model` 辨識** → 新名稱一定落到 fallback 分支，拿不到 `c1`、參數不做寬度縮放，`TypeError: missing 1 required positional argument: 'c2'`。已以別名方式註冊。
7. **`StarTripletBlock` 在 c1≠c2 時必崩** → head 第 19 層輸入 128ch、輸出 32ch，`res + dwconv2(v)` 形狀不符，且 `Conv(c1, c2, g=c1)` 分組不整除。已加入投影捷徑。
8. **backbone 第 7 層 `ADown [512]`** → 官方 yolo26-p2 該層為 `Conv [1024, 3, 2]`，寫成 512 會讓 P5 下採樣出現通道瓶頸（256→128→256）。已改為 `[1024]`。
9. **第 2 / 4 層通道與 v8 不符** → 初稿寫 `[128]`/`[256]`（32/64ch），v8 為 `[256]`/`[512]`（64/128ch）。已對齊，否則 backbone 寬度減半會混淆歸因。
10. **`ratio` 參數未被使用** → 原 `1 - inter/min(area)` 是重疊係數而非 Inner-IoU。已改為論文原版（依 ratio 縮放輔助框後計算 IoU）。
11. **WIoU 未 detach 且非 v3** → 已改為 v3 的離群度聚焦係數，並對外接框對角線 detach。

### 本機驗證
`Train Code/verify_v9_local.py` 是 Step.2 + 3 + 4 + 4.5 的合併版，可在本機（無 GPU 亦可）跑完整驗證：

```
.venv/Scripts/python.exe "Train Code/verify_v9_local.py"
```

2026-08-19 於 ultralytics 8.4.121 / torch 2.13.0+cpu 實測，7 項檢查全數通過。

# Step.1 環境檢查與資料集安全準備
### 檢查GPU、安裝環境，並將唯讀資料集複製至可寫入工作區。

In [ ]:
# 1. 檢查 GPU 狀態
!nvidia-smi
# 2. 安裝套件。版本必須釘死：整套注入機制建立在 8.4.121 的
#    parse_model 與 BboxLoss 實作細節上，換版本可能靜默失效（問題 C6）。
#    tensorflow 已移除 —— 交付只需要 PyTorch 權重，那行在 Kaggle 上白花安裝時間。
!pip install -q ultralytics==8.4.121 pyyaml

In [ ]:
import os
import shutil
import sys
import yaml
import ultralytics
print(f"▷ Ultralytics 版本: {ultralytics.__version__}")

In [ ]:
# 1. 定義進度條渲染與資料集複製校驗函式
def render_progress_bar(current, total, task_name="檔案同步複製中", bar_length=25):
    percent = (current / total) * 100 if total > 0 else 100.0
    filled_length = int(bar_length * current // total) if total > 0 else bar_length
    bar = '█' * filled_length + '░' * (bar_length - filled_length)
    
    # 輸出兩行格式：顯示步驟名稱與進度條
    sys.stdout.write(f"\r▷ 正在執行 [{task_name}] | 進度: [{bar}] {percent:5.1f}% ({current}/{total})")
    sys.stdout.flush()
def copy_and_verify_dataset(src_dir, dst_dir):
    if not os.path.exists(src_dir):
        print(f"▷ 錯誤：找不到來源資料集目錄 {src_dir}")
        return False

    # 步驟 1: 收集來源端所有檔案路徑
    src_files = []
    for root, _, files in os.walk(src_dir):
        for file in files:
            rel_path = os.path.relpath(os.path.join(root, file), src_dir)
            src_files.append(rel_path)
    
    total_files = len(src_files)
    print(f"▷ 來源資料集掃描完成，共計 {total_files} 個檔案")

    # 步驟 2: 逐檔複製並動態刷新進度條
    for idx, rel_path in enumerate(src_files, 1):
        src_path = os.path.join(src_dir, rel_path)
        dst_path = os.path.join(dst_dir, rel_path)
        
        os.makedirs(os.path.dirname(dst_path), exist_ok=True)
        shutil.copy2(src_path, dst_path)
        
        # 每 100 筆或最後一筆刷新終端機顯示
        if idx % 100 == 0 or idx == total_files:
            render_progress_bar(idx, total_files, task_name="▷ 檔案同步複製中")
    
    print("\n\n▷ 正在檢查複製檔案")

    # 步驟 3: 複製後檢查機制（比對檔案存在性與 Byte 大小）
    missing_files = []
    corrupted_files = []
    
    dst_files_set = set()
    for root, _, files in os.walk(dst_dir):
        for file in files:
            rel_path = os.path.relpath(os.path.join(root, file), dst_dir)
            dst_files_set.add(rel_path)

    for rel_path in src_files:
        if rel_path not in dst_files_set:
            missing_files.append(rel_path)
        else:
            src_sz = os.path.getsize(os.path.join(src_dir, rel_path))
            dst_sz = os.path.getsize(os.path.join(dst_dir, rel_path))
            if src_sz != dst_sz:
                corrupted_files.append(rel_path)

    # 步驟 4: 輸出校驗結果報告
    print("≡" * 60)
    print("▷ 資料集複製完整性校驗：")
    print(f"  ▶ 來源檔案總數 (Source)     : {len(src_files)}")
    print(f"  ▶ 目標檔案總數 (Destination): {len(dst_files_set)}")
    print(f"  ▶ 遺漏檔案數   (Missing)    : {len(missing_files)}")
    print(f"  ▶ 損毀/大小不符(Corrupted)  : {len(corrupted_files)}")
    
    if not missing_files and not corrupted_files:
        print("▷ 檢查通過")
        print("≡" * 60)
        return True
    else:
        print("▷ 檢查失敗")
        if missing_files:
            print(f"▷ 遺漏檔案: {missing_files[:5]}")
        if corrupted_files:
            print(f"▷ 損毀檔案: {corrupted_files[:5]}")
        print("≡" * 60)
        return False

In [ ]:
# 2. 執行複製與路徑配置
src_dataset_dir = "/kaggle/input/datasets/yentsai9183/datasets-yolo26-v5"
dst_dataset_dir = "/kaggle/working/datasets-yolo26-v5"

# 執行複製與校驗
is_success = copy_and_verify_dataset(src_dataset_dir, dst_dataset_dir)

if is_success:
    # 3. 更新 data.yaml 路徑
    new_yaml_path = "/kaggle/working/data.yaml"
    orig_yaml_path = f"{dst_dataset_dir}/data.yaml"
    
    if os.path.exists(orig_yaml_path):
        with open(orig_yaml_path, 'r', encoding='utf-8') as f:
            yaml_data = yaml.safe_load(f)
        
        yaml_data['path'] = dst_dataset_dir
        yaml_data['train'] = "train/images"
        yaml_data['val'] = "valid/images"
        yaml_data['test'] = "test/images"
        
        with open(new_yaml_path, 'w', encoding='utf-8') as f:
            yaml.safe_dump(yaml_data, f, default_flow_style=False)
        print(f"▷ data.yaml 已更新\n▷ 新設定檔路徑為: {new_yaml_path}")

    # 4. 清理 BOM 標記與舊快取
    print("▷ 正在刪除BOM標籤和Cache")
    modified_count = 0
    deleted_cache_count = 0
    
    for subdir, _, files in os.walk(dst_dataset_dir):
        if "labels" in subdir:
            for file in files:
                if file.endswith('.txt'):
                    file_path = os.path.join(subdir, file)
                    with open(file_path, 'rb') as f:
                        header = f.read(3)
                    if header == b'\xef\xbb\xbf':
                        with open(file_path, 'r', encoding='utf-8-sig') as f:
                            content = f.read()
                        with open(file_path, 'w', encoding='utf-8') as f:
                            f.write(content)
                        modified_count += 1
                        
        for file in files:
            if file.endswith('.cache'):
                os.remove(os.path.join(subdir, file))
                deleted_cache_count += 1
                
    print(f"    ▶ 已自動修正 BOM 檔案數: {modified_count}")
    print(f"    ▶ 已清理舊快取檔案數: {deleted_cache_count}")
    print("▷ Step.1 完成")

# Step.2 加入 Wise-Inner-MPDIoU（定義於 v9_modules.py）

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Step.2 的損失函式與 Step.3 的自訂模組都已移到 v9_modules.py。
#
# 在此之前，同一份 Wise-Inner-MPDIoU 與 StarTripletBlock 同時存在於
# model-train_v9.ipynb、RESUME.ipynb、verify_v9_local.py 三個檔案裡，
# 任何一份漂移都會讓續跑或驗證的模型與訓練時不一致，而且不會報錯（問題 C3）。
# 現在一律從 GitHub 抓同一份，notebook 內不再放第二份定義。
#
# ⚠ Kaggle 的 Notebook 設定裡必須開啟 Internet。
# ══════════════════════════════════════════════════════════════════════════
import subprocess
import sys

URL = ("https://raw.githubusercontent.com/DreamOver9183/"
       "AY2026_Citrus_Pests_and_Diseases_Project/main/Train%20Code/v9/v9_modules.py")
subprocess.run(["curl", "-sSLf", "-o", "/kaggle/working/v9_modules.py", URL], check=True)

sys.path.insert(0, "/kaggle/working")
import v9_modules as v9

import ultralytics
assert v9.REQUIRED_ULTRALYTICS == ultralytics.__version__, \
    f"v9_modules 要求 ultralytics {v9.REQUIRED_ULTRALYTICS}，實際 {ultralytics.__version__}"
print(f"▷ v9_modules {v9.__version__} 已載入")


# ── 安裝 Wise-Inner-MPDIoU ────────────────────────────────────────────
# 定義在 v9_modules.py。相對於最初版本，內含三個數值修正：
#   1. MPD 分母 detach —— 移除「把外接框撐大來降低損失」的退化梯度路徑
#   2. 全程 fp32 計算 —— AMP 之下 c_diag_sq 在特徵圖尺度就會逼近 fp16 上限 65504
#   3. 離群度統計量留在 GPU 且可存取 —— 免去每個 batch 的 .item() 同步，並讓續跑能還原
import torch
import ultralytics.utils.loss

v9.install_loss(ratio=0.7)          # v9 的原始設定；A1b 那條路線用 1.25
assert ultralytics.utils.loss.bbox_iou.__name__ == "bbox_wise_inner_mpdiou"

# 自我檢查：用與 utils/loss.py:133 完全相同的呼叫方式驗證簽名與形狀
_p = torch.tensor([[10.0, 10.0, 30.0, 30.0], [0.0, 0.0, 10.0, 10.0]], requires_grad=True)
_t = torch.tensor([[12.0, 12.0, 32.0, 32.0], [50.0, 50.0, 60.0, 60.0]])
_iou = ultralytics.utils.loss.bbox_iou(_p, _t, xywh=False, CIoU=True)
assert _iou.shape == (2, 1), f"回傳形狀必須為 [N,1]，實際為 {tuple(_iou.shape)}"
assert torch.isfinite(_iou).all(), "回傳值含 NaN/Inf"
(1.0 - _iou).sum().backward()
assert torch.isfinite(_p.grad).all(), "梯度含 NaN/Inf"
print(f"▷ 簽名/形狀/梯度檢查通過  loss(重疊框)={float((1 - _iou[0]).detach()):.4f}  "
      f"loss(不相交框)={float((1 - _iou[1]).detach()):.4f}")
print("▷ Step.2 完成")

# Step.3 註冊 StarTripletBlock（定義於 v9_modules.py）

In [ ]:
# ── 註冊 StarTripletBlock（ADown 沿用 Ultralytics 內建版）──────────────
# 類別定義在 v9_modules.py。註冊會把 StarTripletBlock 綁到 `C2f` 這個既有名稱，
# 因為 parse_model 的 base_modules / repeat_modules 名單寫死，新名稱拿不到
# c1 注入與 width/depth 縮放。yaml 中出現的 C2f 一律代表 StarTripletBlock。
import torch
import ultralytics.nn.tasks

v9.install_modules()
assert ultralytics.nn.tasks.C2f is v9.StarTripletBlock, "C2f 別名未生效"

# 形狀自我檢查（涵蓋 c1==c2 與 c1!=c2 兩種情形）
assert v9.StarTripletBlock(96, 32, n=1)(torch.randn(2, 96, 32, 32)).shape == (2, 32, 32, 32)
assert v9.StarTripletBlock(32, 32, n=2)(torch.randn(2, 32, 32, 32)).shape == (2, 32, 32, 32)
assert ultralytics.nn.tasks.ADown(32, 64)(torch.randn(2, 32, 32, 32)).shape == (2, 64, 16, 16)
print("▷ StarTripletBlock / ADown 形狀檢查通過")
print("▷ 提醒：日後載入 v9 的 best.pt / last.pt 之前，必須先執行本 cell，")
print("        否則 pickle 找不到自訂類別會報 AttributeError。")
print("▷ Step.3 完成")

# Step.4 產生最佳化的 yolo26-citrus-p2-v9.yaml

In [ ]:
import yaml as _yaml

yaml_v9_content = """
# YOLO26n-Citrus-P2 (v9 Deploy Architecture)
#
# 基準：官方 yolo26-p2.yaml（v8 用的就是這份，未改動）。
# v9 相對 v8 只動三處，其餘層號、通道數、Detect 取用層完全對齊，
# 確保 mAP 差異可歸因到改動本身而非頭部配置：
#   (1) 第 2 / 4 / 19 層  C3k2      → StarTripletBlock
#   (2) 第 3 / 5 / 7 / 20 / 23 / 26 層  stride-2 Conv → ADown
#   (3) box loss  CIoU → Wise-Inner-MPDIoU（Step.2，不在本檔）
#
# ★ 重要：本 yaml 中所有寫 `C2f` 的層，實際建構出來的是 Step.3 定義的
#   StarTripletBlock（StarNet 升維 + Triplet Attention）。
#   原因見 Step.3 註解：parse_model 的 base_modules/repeat_modules 名單寫死，
#   新名稱無法取得 c1 注入與 width/depth 縮放，只能藉由覆寫既有名稱達成。
#   模型摘要表印出的類別名稱仍會是 StarTripletBlock，可據此核對。
#
# ★ end2end / reg_max 必須保留：YOLO26 靠這兩個 key 決定走 NMS-free 的
#   E2EDetectLoss（box + cls + l1）還是舊的 DFL(reg_max=16) 路徑。
#   v8 的 results.csv 有 train/l1_loss 欄位，代表 v8 走的是 E2E/reg_max=1；
#   v9 若漏掉這兩行會靜默變成另一個體系的模型，與 v8 完全無法對照。

nc: 8
end2end: True
reg_max: 1
scales:
  n: [0.50, 0.25, 1024]

# 右側 -> 為 scale=n (depth 0.50 / width 0.25) 縮放後的實際輸出通道數，與 v8 逐層相同
backbone:
  - [-1, 1, Conv, [64, 3, 2]]          # 0-P1/2                    -> 16
  - [-1, 1, Conv, [128, 3, 2]]         # 1-P2/4                    -> 32
  - [-1, 2, C2f, [256]]                # 2-P2/4  StarTripletBlock  -> 64
  - [-1, 1, ADown, [256]]              # 3-P3/8  雙路下採樣        -> 64
  - [-1, 2, C2f, [512]]                # 4-P3/8  StarTripletBlock  -> 128
  - [-1, 1, ADown, [512]]              # 5-P4/16 雙路下採樣        -> 128
  - [-1, 2, C3k2, [512, True]]         # 6-P4/16                   -> 128
  - [-1, 1, ADown, [1024]]             # 7-P5/32 雙路下採樣        -> 256
  - [-1, 2, C3k2, [1024, True]]        # 8-P5/32                   -> 256
  - [-1, 1, SPPF, [1024, 5, 3, True]]  # 9                         -> 256
  - [-1, 2, C2PSA, [1024]]             # 10                        -> 256

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]          # 12  256+128               -> 384
  - [-1, 2, C3k2, [512, True]]         # 13                        -> 128

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]          # 15  128+128               -> 256
  - [-1, 2, C3k2, [256, True]]         # 16 (P3/8)                 -> 64

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 2], 1, Concat, [1]]          # 18  64+64                 -> 128
  - [-1, 2, C2f, [128]]                # 19 (P2/4-xsmall) STB      -> 32

  - [-1, 1, ADown, [128]]              # 20 ADown 取代 stride-2 Conv -> 32
  - [[-1, 16], 1, Concat, [1]]         # 21  32+64                 -> 96
  - [-1, 2, C3k2, [256, True]]         # 22 (P3/8)                 -> 64

  - [-1, 1, ADown, [256]]              # 23                        -> 64
  - [[-1, 13], 1, Concat, [1]]         # 24  64+128                -> 192
  - [-1, 2, C3k2, [512, True]]         # 25 (P4/16)                -> 128

  - [-1, 1, ADown, [512]]              # 26                        -> 128
  - [[-1, 10], 1, Concat, [1]]         # 27  128+256               -> 384
  - [-1, 1, C3k2, [1024, True, 0.5, True]] # 28 (P5/32)            -> 256

  - [[19, 22, 25, 28], 1, Detect, [nc]] # Detect(P2, P3, P4, P5)
"""

YAML_PATH = "/kaggle/working/yolo26-citrus-p2-v9.yaml"
with open(YAML_PATH, "w", encoding="utf-8") as f:
    f.write(yaml_v9_content.strip())

# ── 立即回讀驗證，避免縮排/語法錯誤拖到建模才發現 ────────────────────
with open(YAML_PATH, "r", encoding="utf-8") as f:
    _d = _yaml.safe_load(f)

assert _d["nc"] == 8 and _d["end2end"] is True and _d["reg_max"] == 1, _d
_layers = _d["backbone"] + _d["head"]
assert len(_layers) == 30, f"層數 = {len(_layers)}，應為 30 (0~29)"
assert _layers[-1][2] == "Detect" and _layers[-1][0] == [19, 22, 25, 28], _layers[-1]
print(f"▷ yaml 回讀驗證通過：{len(_layers)} 層，Detect 取用 {_layers[-1][0]}")
print(f"▷ StarTripletBlock(以 C2f 別名)位於: {[i for i, l in enumerate(_layers) if l[2] == 'C2f']}")
print(f"▷ ADown 位於: {[i for i, l in enumerate(_layers) if l[2] == 'ADown']}")
print("▷ Step.4 完成")

# Step.4.5 架構隔離驗證
### 在燒 GPU 時數之前，先確認模型可建構、可前向、可算出有限的損失值。
### 這一格若通過，Step.5 的 1-Epoch 測試才有意義。

In [ ]:
import math
import torch
from ultralytics import YOLO
from ultralytics.cfg import get_cfg

YAML_PATH = "/kaggle/working/yolo26-citrus-p2-v9.yaml"

# ── 1. 建構（ADown / StarTripletBlock 的 shape 或註冊問題會在這裡爆）────
model = YOLO(YAML_PATH)
core = model.model
print("\n▷ 1/5 模型建構成功")

# ── 2. 確認 E2E 體系與 v8 一致 ────────────────────────────────────────
det = core.model[-1]
assert getattr(det, "end2end", False), "Detect 不是 end2end：yaml 少了 end2end: True"
assert det.reg_max == 1, f"reg_max = {det.reg_max}，yaml 少了 reg_max: 1（v8 為 1）"
assert det.nc == 8, f"nc = {det.nc}"
assert det.nl == 4, f"偵測頭數量 = {det.nl}，應為 4 (P2/P3/P4/P5)"
strides = [int(s) for s in det.stride]
assert strides == [4, 8, 16, 32], f"strides = {strides}"
print(f"▷ 2/5 end2end=True, reg_max={det.reg_max}, nc={det.nc}, 偵測頭={det.nl}, strides={strides}")

# ── 3. 確認自訂模組真的被建構出來（yaml 寫 C2f，實際應為 StarTripletBlock）──
types = {i: m.type.split(".")[-1] for i, m in enumerate(core.model)}
for idx in (2, 4, 19):
    assert types[idx] == "StarTripletBlock", f"第 {idx} 層是 {types[idx]}，別名注入沒生效"
for idx in (3, 5, 7, 20, 23, 26):
    assert types[idx] == "ADown", f"第 {idx} 層是 {types[idx]}"
print("▷ 3/5 StarTripletBlock @ [2, 4, 19]、ADown @ [3, 5, 7, 20, 23, 26] 全部就位")

# ── 4. 前向傳播 ───────────────────────────────────────────────────────
core.eval()
with torch.no_grad():
    core(torch.zeros(1, 3, 640, 640))
print("▷ 4/5 Forward pass 成功")

# ── 5. 走一次完整損失路徑（自訂 bbox_iou 的簽名/形狀/NaN 都在這裡驗）──
core.args = get_cfg(overrides={"box": 5.0, "cls": 1.0, "dfl": 1.5})
core.train()
fake_batch = {
    "img": torch.rand(2, 3, 640, 640),
    "batch_idx": torch.tensor([0.0, 0.0, 1.0]),
    "cls": torch.tensor([[4.0], [6.0], [1.0]]),          # Scale_Insect / Thrips / Canker
    "bboxes": torch.tensor([[0.50, 0.50, 0.04, 0.04],    # 刻意用微小框，貼近真實標註分佈
                            [0.22, 0.31, 0.02, 0.03],
                            [0.71, 0.68, 0.15, 0.12]]),
}
loss, loss_items = core.loss(fake_batch)
# loss_items 在不同 Ultralytics 版本可能是 dict 或 3 元素 tensor，兩種都接
if isinstance(loss_items, dict):
    items = {k: float(v) for k, v in loss_items.items()}
else:
    items = {k: float(v) for k, v in zip(("box", "cls", "l1/dfl"), loss_items.flatten())}
assert torch.isfinite(loss).all(), "總損失出現 NaN/Inf"
assert all(math.isfinite(v) for v in items.values()), f"分項損失出現 NaN/Inf: {items}"
print("▷ 5/5 損失路徑通過（含自訂 Wise-Inner-MPDIoU）")
print("      " + "  ".join(f"{k}={v:.4f}" for k, v in items.items()) + f"  | total={float(loss.sum().detach()):.4f}")

print("\n" + "═" * 62)
core.eval()
model.info(detailed=False, verbose=True)      # 參數量 / GFLOPs
print("═" * 62)
print("▷ 本機基準（ultralytics 8.4.121, torch 2.13.0+cpu, imgsz=640, nc=8）：")
print("    v8 官方 yolo26-p2 : 2,517,824 params /  7.65 GFLOPs")
print("    v9 本檔           : 2,225,716 params / 11.00 GFLOPs")
print("  上方 model.info() 的數字若與 v9 這行相差過大，代表 yaml 或模組註冊有出入。")
print("\n▷ Step.4.5 全部通過")
print("▷ 2026-08-19 已於 Kaggle T4 完成 1-Epoch 實測，本次執行應與下列數字相符：")
print("    Transferred 284/998 items（v8 為 360/902，差距來自被換掉的淺層 backbone）")
print("    box_loss 4.409  cls_loss 13.76  l1_loss 0.03339  mAP50 0.0758")
print("    355 s/epoch、VRAM 峰值 14.1G/14.9G @ batch=24")
print("  若本次數字與上列明顯不同，代表 yaml、模組註冊或 loss 有出入，先別開始長跑。")
print("\n▷ 160 epochs 約需 15.8 小時，超過 Kaggle 單場上限（互動式 9h / Save & Run All 12h）；")
print("  Session 1 由 Step.5 的 stop_and_snapshot callback 自動停止，Session 2 用 RESUME.ipynb。")

# Step.5 YOLO26-nano+P2 模型測試與正式訓練
### 設定超參數、先執行1-Epoch 快速測試，再正式啟動完整訓練。

```
# 超參數設定：
# epochs=160                                         # 延長週期以供後期微調
# imgsz=640                                          # 對應 P2 特徵圖 160x160
# batch=20                                           # 批次大小（1-Epoch 實測 batch=24 時 VRAM 峰值 14.1G/14.9G）
# patience=30                                        # 早停容忍度
# plots=True                                         # 評估圖表生成
# optimizer=MuSGD                                    # 優化器：μP 架構微步隨機梯度下降法
# lr0=0.008                                          # 初始學習率：微降步長以穩定 P2 梯度
# lrf=0.01                                           # 最終學習率比例：Cosine 衰減至 lr0*0.01
# momentum=0.937                                     # 動量係數：維持梯度慣性跨越極小值
# cos_lr=True                                        # 餘弦退火排程：突破 Epoch 80 停滯瓶頸
# warmup_epochs=3.0                                  # 預熱輪次：前 3 輪平滑升溫保護預訓練權重
# mosaic=0.7                                         # 拼圖增強機率：減少微小害蟲被縮至亞像素等級
# close_mosaic=30                                    # 關閉拼圖：最後 30 輪停用 Mosaic 微調邊界
# box=5.0                                            # 邊界框損失權重：對應等效權重約 10.8（見下方說明）
# cls=1.0                                            # 分類損失權重：拉升薊馬/蚜蟲/潛葉蛾召回率
# dfl=1.5                                            # 分佈焦點損失權重（reg_max=1，實際作用於 L1 分支）
# cache=ram                                          # 記憶體快取：常駐 RAM 消除磁碟 I/O 延遲
# workers=4                                          # 資料載入線程數：匹配 Kaggle vCPU
# device=0                                           # 運算裝置：指定第 0 號 GPU
# name=YOLO26n_P2_Citrus_MuSGD_v9                    # 專案名稱
```

### 與 v8 的差異
| 項目 | v8 | v9 |
| --- | --- | --- |
| `batch` | 32 | 20（實測 batch=24 峰值 14.1G/14.9G，16 小時的長跑需要餘裕） |
| `cls` | 0.8 | 1.0（依 v8 報告建議壓制介殼蟲背景 FP） |
| `box` | 8.0 | 5.0（見下方量級說明） |
| box loss | 內建 CIoU | Wise-Inner-MPDIoU（Step.2） |
| 第 2 / 4 / 19 層 | `C3k2` | `StarTripletBlock` |
| 第 3 / 5 / 7 / 20 / 23 / 26 層 | stride-2 `Conv` | `ADown` |
| 偵測頭 | P2/P3/P4/P5 | P2/P3/P4/P5（不變） |
| `end2end` / `reg_max` | True / 1 | True / 1（不變） |

### `box` 由 8.0 改為 5.0 的依據
本機實測（同一批微小框，`verify_v9_local.py`）Wise-Inner-MPDIoU 的損失均值是內建 CIoU 的
**2.16 倍**——因為它是 `WIoU + λ_mpd·MPD + λ_inner·(1-InnerIoU)` 三項相加。
若沿用 v8 的 `box=8.0`，實際等效權重會變成 **17.3**，box / cls / l1 的平衡與 v8 差距過大。

`box=5.0` 對應等效權重約 **10.8**，相對 v8 適度加強定位（符合 v9 針對微小病蟲害的意圖），
又不會把 cls 壓過頭。若要完全對齊 v8 的等效權重（8.0），改用 `box=3.7`；
或維持 `box=8.0` 而把 Step.2 的 `lambda_mpd` / `lambda_inner` 預設值降到 0.5（等效約 12.4）。

1-Epoch 測試後仍請比對 `train/box_loss` 與 v8 第 1 輪的 `3.11981`，確認在同一數量級。

### ⚠ 一次改三件事的歸因問題
v9 同時更動了 loss、ADown、StarTripletBlock。若最終指標不如 v8，
無法判斷是哪一項造成的。若時間允許，建議的消融順序（各跑 1 次完整訓練）：
1. v9a = v8 架構 + 僅換 loss
2. v9b = v9a + ADown
3. v9c = v9b + StarTripletBlock（即目前的 v9）

## YOLO26 Nano+P2, 1 Epoch 測試

In [ ]:
from ultralytics import YOLO
# 載入權重
model = YOLO("/kaggle/working/yolo26-citrus-p2-v9.yaml")
model.load("yolo26n.pt") 
# 超參數設定
results = model.train(
    data="/kaggle/working/data.yaml",
    name="YOLO26n_P2_Citrus_MuSGD_v9",
    plots=True,
    device=0,
    # 週期與解析度
    epochs=1,
    imgsz=640,
    batch=20,
    patience=30,
    # 硬體設定
    cache="ram",
    workers=4,
    # 優化器設定
    optimizer="MuSGD",
    lr0=0.008,
    lrf=0.01,
    momentum=0.937,
    cos_lr=True,
    warmup_epochs=3.0,
    # loss 權重設定
    mosaic=0.7,               
    close_mosaic=30,          
    box=5.0,                  # v8 為 8.0；Wise-Inner-MPDIoU 量級為 CIoU 的 2.16 倍，
                              # 5.0 對應等效權重約 10.8（見 Step.0 損失量級表）
    cls=1.0,
    dfl=1.5
)
print("▷ Step.5 測試成功")

## YOLO26 Nano+P2, 正式訓練

In [ ]:
import shutil
import time

import ultralytics.utils.loss
from ultralytics import YOLO

# ═══════════════════════════════════════════════════════════════════════
# Session 1：跑到 STOP_AFTER_EPOCHS 後乾淨停止，剩餘輪次交給 RESUME.ipynb
#
# Kaggle 單場上限：互動式 9 小時 / Save & Run All 12 小時。
# 355 s/epoch 是 batch=24 量到的；batch=20 的 iteration 數多 20%，
# 實際 s/epoch 會更高，所以「105 輪 ≈ 10.8 小時」這個推算並不可靠。
# 現在改以 DEADLINE_HOURS 為主要安全閥，輪數上限只是次要條件 ——
# 先到哪個就停哪個，不必事先猜準 s/epoch。
# ═══════════════════════════════════════════════════════════════════════
STOP_AFTER_EPOCHS = 105
DEADLINE_HOURS    = 10.5   # Save & Run All 上限 12 h，留 1.5 h 餘裕

model = YOLO("/kaggle/working/yolo26-citrus-p2-v9.yaml")
model.load("yolo26n.pt")


def stop_and_snapshot(trainer):
    """每輪保留可續跑的 checkpoint，並在輪數/時數上限時乾淨停止。

    訓練迴圈結束後一定會執行 final_eval() → strip_optimizer()，把 last.pt / best.pt
    的 epoch 改寫成 -1 並清掉 optimizer / EMA / scaler，那種檔案無法續跑
    （resume_training() 會在 `assert 0 < start_epoch < self.epochs` 失敗）。
    on_fit_epoch_end 的觸發點在 save_model() 之後、跳出迴圈之前
    （engine/trainer.py 的 610 / 624 / 633 行），此時 last.pt 才剛寫好且尚未被 strip。

    兩處修正：
      1. 備份不設條件。patience 早停時 trainer.stop 在 trainer.py:605 就已為 True，
         原本寫成 `if not trainer.stop` 會讓整段被跳過，等於在最需要救援時失效（問題 C2）。
      2. 加上牆鐘上限。原本的 STOP_AFTER_EPOCHS=105 是用 batch=24 量到的
         355 s/epoch 推的；batch=20 的 iteration 數多 20%，實際會逼近 Kaggle
         Save & Run All 的 12 h 硬上限，一旦超時整場輸出不保留（問題 C1）。
    """
    if trainer.last.exists():
        shutil.copy(trainer.last, trainer.wdir / "resume_from.pt")
    # 自訂 loss 的離群度統計量不在 checkpoint 裡，續跑要靠這份側錄還原
    if ultralytics.utils.loss.bbox_iou.__name__ == "bbox_wise_inner_mpdiou":
        (trainer.wdir / "wiou_state.txt").write_text(str(v9.wiou_state()))

    elapsed = (time.time() - trainer.train_time_start) / 3600
    if not trainer.stop and (trainer.epoch + 1 >= STOP_AFTER_EPOCHS
                             or elapsed > DEADLINE_HOURS):
        trainer.stop = True
        print(f"\n▷ 停於第 {trainer.epoch + 1} / {trainer.epochs} 輪，已耗時 {elapsed:.2f} h")
        print(f"▷ 續跑用 checkpoint 已保留：{trainer.wdir / 'resume_from.pt'}")
        print("▷ 接著執行 Step.6 壓出 runs.zip，下載後上傳成 Kaggle Dataset，改用 RESUME.ipynb")


model.add_callback("on_fit_epoch_end", stop_and_snapshot)

# 超參數設定
results = model.train(
    data="/kaggle/working/data.yaml",
    name="YOLO26n_P2_Citrus_MuSGD_v9",
    plots=True,
    device=0,
    # 週期與解析度
    epochs=160,               # 必須維持 160：resume 會沿用 ckpt 內記錄的總輪數
    imgsz=640,
    batch=20,
    patience=30,
    save_period=10,           # 每 10 輪留一份未被 strip 的 epochN.pt，作為意外中斷的保險
    # 硬體設定
    cache="ram",
    workers=4,
    # 優化器設定
    optimizer="MuSGD",
    lr0=0.008,
    lrf=0.01,
    momentum=0.937,
    cos_lr=True,
    warmup_epochs=3.0,
    # loss 權重設定
    mosaic=0.7,
    close_mosaic=30,
    box=5.0,                  # v8 為 8.0；Wise-Inner-MPDIoU 量級為 CIoU 的 2.16 倍，
                              # 5.0 對應等效權重約 10.8（見 Step.0 損失量級表）
    cls=1.0,
    dfl=1.5
)
print("▷ Step.5 訓練完畢")

## RESUME 訓練

續跑已獨立成 **`RESUME.ipynb`**，請在第二場 Kaggle session 開該檔執行，本檔不需要再跑。

1-Epoch 實測為 355 s/epoch，160 epochs 約 **15.8 小時**，一場 session 跑不完，因此拆成兩場。

### Kaggle 單場上限與建議切分
| 執行方式 | 單場上限 | Session 1 建議輪數 | 實際耗時 | Session 2 |
| --- | --- | --- | --- | --- |
| **Save & Run All** | 12 小時 | `STOP_AFTER_EPOCHS = 105` | 約 10.8 小時 | 55 輪，約 5.7 小時 |
| 互動式 | 9 小時 | `STOP_AFTER_EPOCHS = 80` | 約 8.3 小時 | 80 輪，約 8.3 小時 |

**建議用 Save & Run All**：可全程無人看顧，且兩場都留有餘裕。
Kaggle 每週 GPU 配額 30 小時，兩場合計約 17 小時，足夠。

### 為什麼要靠 callback 停，而不是等時間到
Ultralytics 的訓練迴圈一結束就會執行 `final_eval()` → `strip_optimizer()`，
把 `last.pt` / `best.pt` 的 `epoch` 改寫成 **-1** 並清掉 optimizer / EMA / scaler，
這種檔案無法續跑（`resume_training()` 會在 `assert 0 < start_epoch < self.epochs` 失敗）。

Step.5 的 `stop_and_snapshot` callback 在 `on_fit_epoch_end` 觸發 —— 剛好在 `save_model()`
之後、跳出迴圈之前 —— 先把當下的 `last.pt` 複製成 **`resume_from.pt`** 再喊停，
所以那份檔案不會被 strip。另外 `save_period=10` 每 10 輪留一份未被 strip 的 `epochN.pt`，
萬一 session 被意外中斷也還有退路。

> Save & Run All 若跑超過 12 小時會**整場失敗且輸出不保留**，所以務必讓 callback 先停下來。

### Session 1 結束後要做的事
執行 **Step.6** 壓出 `runs.zip`，下載後上傳成一個 Kaggle Dataset，
再於 `RESUME.ipynb` 的 Step.R 填入該 Dataset 路徑。

In [ ]:
# 續跑請改用同目錄的 RESUME.ipynb，本 cell 保留為佔位說明。
#
# 為什麼不能只在這裡加一行 model.train(resume=True)：
#   1. last.pt 內含 pickle 過的 StarTripletBlock / StarBlock / TripletAttention，
#      新 session 必須先重跑 Step.2 與 Step.3 才載得回來（否則 AttributeError）。
#   2. 只還原 last.pt 不夠，results.csv 會從續跑的那一輪重新建檔，
#      最後做不出完整 160 輪的訓練曲線；RESUME.ipynb 會還原整個 runs 目錄。
print("▷ 請開啟 RESUME.ipynb")

# Step.6 Output整理

In [ ]:
import os
import shutil

# 定義工作區與 uns目錄路徑
working_dir = "/kaggle/working"
runs_dir = os.path.join(working_dir, "runs")
zip_output_path = os.path.join(working_dir, "runs")

# 檢查runs目錄並壓縮
if os.path.exists(runs_dir):
    print("▷ 正在壓縮訓練輸出")
    shutil.make_archive(zip_output_path, 'zip', runs_dir)
    
    zip_full_path = f"{zip_output_path}.zip"
    if os.path.exists(zip_full_path):
        size_mb = os.path.getsize(zip_full_path) / (1024 * 1024)
        print(f"▷ 壓縮成功 {zip_full_path} ({size_mb:.2f} MB)")
else:
    print(f"▷ 壓縮失敗: 找不到 '{runs_dir}' 資料夾")